# TPC-C Collector Demo

# Missing
* connection data beyond loading
* join to connection data
* join to monitoring_single
* connection and configuration also in monitoring and connection df

## Naming
* There is an experiment having a code, say `1775855486`.
* The experiment inspects a SUT, say `PostgreSQL-A`. This is called a `configuration`.
* The experiment is run several times, say twice. The indicator of the run is called `experiment_run`.
* Each run can have several phases as a sequence. The number of the phase is called `client`. The state of the configuration in a phase is called a `connection`.
* Each client can have several `pods`, that are run in parallel. A pod represents a driver.
* Performance metrics are collected per driver pod.  
    The naming of an instance is `<sut>-<experiment_run>-<client>-<pod>`. It is unique per experiment.
* Monitoring metrics are collected per phase. They are automatically aggregated across parallel pods.  
    The naming of an instance is `<sut>-<experiment_run>-<client>`. It is unique per experiment.

## Aggregation
* Aggregation is complicated. Some metrics are aggregated via count, sum, max or average. Others cannot be aggregated sensibly, like experiment code or latency percentiles.
* There are helper functions to aggregated pods that are certainly run in parallel.  
  So `<sut>-<experiment_run>-<client>-<pod>` are aggregated to `<sut>-<experiment_run>-<client>`.
* An exception is multi-tenancy.

## Class `collector`
* constructor `collectors.benchbase(path, codes)`
* evaluator object `evaluate = collect.get_evaluator()`
* dataframe of connection infos `collect.get_connections()`
  * index is name of client
* **monitoring metrics**
  * dataframe of available metrics `collect.df_metrics`
    * index is key of metric
  * dataframe of monitored components `collect.get_monitored_phases()`
    * index is key of component
  * dataframe of aggregated metrics per connection `collect.get_monitoring_single_all()`
    * index is name of client
    * metrics aggregated per code, experiment_run and client
  * dataframe of aggregated metrics per connection and across tenants `collect.get_monitoring_all()`
    * index just enumerates
    * metrics aggregated per code, experiment_run and client and across tenants
  * dataframe of time series for a metric of a connection in an experiment `collect.get_monitoring_timeseries_single(code, metric)`
    * index is name of connection
    * unstacked (wide format)
  * dataframe of time series for a metric in all experiments `collect.get_monitoring_timeseries_all(metric)`
    * index just enumerates
    * stacked (long format)
* **performance metrics**
  * dataframe of loading metrics `collect.get_loading_time_max_all()`
    * index is name of client
  * dataframe of performance aggregated per parallel client `collect.get_performance_all()`
    * index just enumerates
    * performance aggregated per code, experiment_run and client
  * dataframe of performance for one experiment `collect.get_performance_single()`
    * performance of each single client (driver)
    * index is name of client pod
  * dataframe of performance for all experiments `collect.get_performance_all_single()`
    * performance of each single client (driver)
    * index is name of client pod


[1] [Benchmarking Multi-Tenant Architectures in PostgreSQL](https://doi.org/10.48786/edbt.2026.46)
> Erdelt, P.K., and Rabl T. (2026)
> In: Proceedings 29th International Conference on Extending Database Technology, EDBT 2026
> OpenProceedings.org
> https://doi.org/10.48786/edbt.2026.46


In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.float_format = '{:.2f}'.format

from bexhoma import collectors

%matplotlib inline

# Functions for Nice Plots

In [2]:
from notebookutils import *

# Collect Results

In [3]:
#path = r"D:\data\benchmarks"
path = r"/data/benchmarks"
filename_prefix = "demo_"
b_plot_save = False
b_skip_plots = True

In [4]:
codes = [
    "1776678767",
    "1776682364"
]

codes

['1776678767', '1776682364']

In [5]:
collect = collectors.ycsb(path, codes)

# Get all Metrics Metadata

## Metrics Names and Types

Metrics that are derived from monitoring

In [6]:
collect.df_metrics

,title,active,type,metric
total_cpu_memory,Memory Usage [MiB],True,cluster,gauge
total_cpu_memory_cached,Memory Usage Cached [MiB],True,cluster,gauge
total_cpu_util,CPU Utilization,True,cluster,gauge
total_cpu_throttled,CPU Throttle,True,cluster,gauge
total_cpu_throttled_s,CPU Throttled Time [s],True,cluster,counter
total_cpu_util_others,CPU Utilization Others,False,cluster,gauge
total_cpu_util_s,CPU Utilization Time [s],True,cluster,counter
total_cpu_util_user_s,CPU User Time [s],True,cluster,counter
total_cpu_util_sys_s,CPU System Time [s],True,cluster,counter
total_cpu_util_others_s,CPU Utilization Time Others [s],False,cluster,counter


## Names of Monitored Components

Names of components and their phases

In [7]:
collect.get_monitored_components()

,description
loading,Loading phase: SUT deployment
loader,Loading phase: component loader
stream,Execution phase: SUT deployment
benchmarker,Execution phase: component benchmarker


# Get Connection Infos

List of states of SUTs, containing loading infos.

In [8]:
df=collect.get_connections()

In [9]:
df.T

,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-4,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-8
code,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364
connection,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-4,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-8
configuration,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2
experiment_run,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
client,1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2
dockerimage,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3
time_load,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00
time_ingest,886.00,886.00,886.00,886.00,886.00,886.00,886.00,886.00,886.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00
time_check,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00


In [10]:
df[['phase', 'code', 'connection', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']]

,phase,code,connection,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-1,1776678767,1776678767-PostgreSQL-64-8-65536-1-1,PostgreSQL-64-8-65536,1,1,None,0,False
1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-1,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-2,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-3,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-4,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-5,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-6,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-7,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-2-8,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536-2-8,PostgreSQL-64-8-65536,1,2,None,0,False
1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-1,1776682364,1776682364-PostgreSQL-64-8-65536-1-1,PostgreSQL-64-8-65536,1,1,None,0,False


In [11]:
collectors.get_non_constant(df).T

,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-4,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-8
code,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364
connection,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-4,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-8
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2
client,1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2
time_load,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00
time_ingest,886.00,886.00,886.00,886.00,886.00,886.00,886.00,886.00,886.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00
time_check,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00
pods,1,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8
datadisk,7091,8548,8548,8548,8548,8548,8548,8548,8548,7091,8546,8546,8546,8546,8546,8546,8546,8546
host_disk,158125,159582,159582,159582,159582,159582,159582,159582,159582,158125,159581,159581,159581,159581,159581,159581,159581,159581


# Monitoring Aggregated per Phase

In [12]:
df = collect.get_monitoring_aggregated_per_phase("stream")
df.T

,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
Memory Usage [MiB],10777.55,11361.35,10771.38,11360.51
Memory Usage Cached [MiB],14783.39,15917.04,14779.38,15918.08
CPU Utilization,0.90,0.85,0.91,0.86
CPU Throttle,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0.00,0.00,0.00,0.00
CPU Utilization Time [s],951.91,901.84,959.41,920.40
CPU User Time [s],482.45,441.85,494.89,450.35
CPU System Time [s],469.46,459.98,464.52,470.05
Network Rx Total [MiB],2058.80,1879.22,2203.71,1839.69
Network Tx Total [MiB],3198.48,1875.98,3322.41,1867.74


### With Metadata

In [13]:
df = collect.add_metadata(df)
df.T

combine on index and column 'phase'


phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
Memory Usage [MiB],10777.55,11361.35,10771.38,11360.51
Memory Usage Cached [MiB],14783.39,15917.04,14779.38,15918.08
CPU Utilization,0.90,0.85,0.91,0.86
CPU Throttle,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0.00,0.00,0.00,0.00
CPU Utilization Time [s],951.91,901.84,959.41,920.40
CPU User Time [s],482.45,441.85,494.89,450.35
CPU System Time [s],469.46,459.98,464.52,470.05
Network Rx Total [MiB],2058.80,1879.22,2203.71,1839.69
Network Tx Total [MiB],3198.48,1875.98,3322.41,1867.74


In [14]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].T

phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
code,1776678767,1776678767,1776682364,1776682364
configuration,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536
experiment_run,1,1,1,1
client,1,2,1,2
type_tenants,None,None,None,None
num_tenants,0,0,0,0
vol_tenants,False,False,False,False


# Time Series of a Single Metric for a Single Experiment

In [15]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_per_phase(code, metric=metric, component='stream')
df.T

,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2
0,9579.14,10350.32
1,9579.14,10350.32
2,9579.14,10350.32
3,9579.14,10350.32
4,9790.00,10350.32
5,9790.00,10350.32
6,9790.00,10350.32
7,9790.00,10350.32
8,9790.00,10350.32
9,9790.00,10350.32


### With Metadata

In [16]:
df = collect.add_metadata(df)
df.T

combine on index and column 'phase'


phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2
0,9579.14,10350.32
1,9579.14,10350.32
2,9579.14,10350.32
3,9579.14,10350.32
4,9790.00,10350.32
5,9790.00,10350.32
6,9790.00,10350.32
7,9790.00,10350.32
8,9790.00,10350.32
9,9790.00,10350.32


In [17]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].head()

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
phase,,,,,,,,
1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-1,1776678767,PostgreSQL-64-8-65536,1,1,None,0,False
1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767,PostgreSQL-64-8-65536,1,2,None,0,False


# Time Series of a Single Metric for All Experiments

In [18]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_all(metric=metric)
df.head()

,timestamp,code,phase,experiment_run,client,type_tenants,vol_tenants,num_tenants,metric,component,value
0,0,1776678767,1776678767-PostgreSQL-64-8-65536-1,1,1,None,False,0,total_cpu_memory,stream,9579.14
1,0,1776678767,1776678767-PostgreSQL-64-8-65536-2,1,2,None,False,0,total_cpu_memory,stream,82802.53
2,0,1776682364,1776682364-PostgreSQL-64-8-65536-1,1,1,None,False,0,total_cpu_memory,stream,9506.29
3,0,1776682364,1776682364-PostgreSQL-64-8-65536-2,1,2,None,False,0,total_cpu_memory,stream,82789.47
4,1,1776678767,1776678767-PostgreSQL-64-8-65536-1,1,1,None,False,0,total_cpu_memory,stream,9579.14


In [19]:
df[['timestamp', 'phase', 'value']].head(8)

,timestamp,phase,value
0,0,1776678767-PostgreSQL-64-8-65536-1,9579.14
1,0,1776678767-PostgreSQL-64-8-65536-2,82802.53
2,0,1776682364-PostgreSQL-64-8-65536-1,9506.29
3,0,1776682364-PostgreSQL-64-8-65536-2,82789.47
4,1,1776678767-PostgreSQL-64-8-65536-1,9579.14
5,1,1776678767-PostgreSQL-64-8-65536-2,82802.53
6,1,1776682364-PostgreSQL-64-8-65536-1,9506.33
7,1,1776682364-PostgreSQL-64-8-65536-2,82789.47


In [20]:
df=collect.add_metadata(df)
df = df.sort_values(['timestamp', 'phase'])
df[['timestamp', 'phase', 'value']].head(8)

combine on columns phase


,timestamp,phase,value
1776678767-PostgreSQL-64-8-65536-1,0,1776678767-PostgreSQL-64-8-65536-1,9579.14
1776678767-PostgreSQL-64-8-65536-2,0,1776678767-PostgreSQL-64-8-65536-2,82802.53
1776682364-PostgreSQL-64-8-65536-1,0,1776682364-PostgreSQL-64-8-65536-1,9506.29
1776682364-PostgreSQL-64-8-65536-2,0,1776682364-PostgreSQL-64-8-65536-2,82789.47
1776678767-PostgreSQL-64-8-65536-1,1,1776678767-PostgreSQL-64-8-65536-1,9579.14
1776678767-PostgreSQL-64-8-65536-2,1,1776678767-PostgreSQL-64-8-65536-2,82802.53
1776682364-PostgreSQL-64-8-65536-1,1,1776682364-PostgreSQL-64-8-65536-1,9506.33
1776682364-PostgreSQL-64-8-65536-2,1,1776682364-PostgreSQL-64-8-65536-2,82789.47


In [21]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].head()

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-1,1776678767,PostgreSQL-64-8-65536,1,1,None,0,False
1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767,PostgreSQL-64-8-65536,1,2,None,0,False
1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-1,1776682364,PostgreSQL-64-8-65536,1,1,None,0,False
1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364,PostgreSQL-64-8-65536,1,2,None,0,False
1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-1,1776678767,PostgreSQL-64-8-65536,1,1,None,0,False


## Custom Aggregation and Scale

In [22]:
metric = 'total_cpu_util'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["code", "experiment_run", "client", "type_tenants", "num_tenants"])["value"]
      .max()
      .reset_index()
)
#plot_bars(df_agg, y='value', title='Max CPU Utilization [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)
df_agg.index.name = 'Max ' + collect.df_metrics.loc[metric]['title']
df_agg

,code,experiment_run,client,type_tenants,num_tenants,value
Max CPU Utilization,,,,,,
0,1776678767,1,1,None,0,1.26
1,1776678767,1,2,None,0,7.53
2,1776682364,1,1,None,0,1.36
3,1776682364,1,2,None,0,7.56


# Performance Results per Connection

In [23]:
df_performance = collect.get_performance_per_connection()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

connection_pod,1776678767-PostgreSQL-64-8-65536-1-1-1,1776678767-PostgreSQL-64-8-65536-2-4-1,1776678767-PostgreSQL-64-8-65536-2-2-1,1776678767-PostgreSQL-64-8-65536-2-8-1,1776678767-PostgreSQL-64-8-65536-2-1-1,1776678767-PostgreSQL-64-8-65536-2-7-1,1776678767-PostgreSQL-64-8-65536-2-5-1,1776678767-PostgreSQL-64-8-65536-2-6-1,1776678767-PostgreSQL-64-8-65536-2-3-1,1776682364-PostgreSQL-64-8-65536-1-1-1,1776682364-PostgreSQL-64-8-65536-2-5-1,1776682364-PostgreSQL-64-8-65536-2-8-1,1776682364-PostgreSQL-64-8-65536-2-1-1,1776682364-PostgreSQL-64-8-65536-2-3-1,1776682364-PostgreSQL-64-8-65536-2-7-1,1776682364-PostgreSQL-64-8-65536-2-6-1,1776682364-PostgreSQL-64-8-65536-2-2-1,1776682364-PostgreSQL-64-8-65536-2-4-1
code,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2
connection,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-8,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-4
configuration,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536
experiment_run,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
client,1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2
pod,g7bjq.dbmsbenchmarker,qf4pk.dbmsbenchmarker,khrjg.dbmsbenchmarker,tn66p.dbmsbenchmarker,rjcxw.dbmsbenchmarker,pr5fr.dbmsbenchmarker,97hvx.dbmsbenchmarker,944k6.dbmsbenchmarker,t9mf8.dbmsbenchmarker,jqtfk.dbmsbenchmarker,66kwz.dbmsbenchmarker,vmhc7.dbmsbenchmarker,8q8zp.dbmsbenchmarker,pctgl.dbmsbenchmarker,tp8ct.dbmsbenchmarker,kc9n2.dbmsbenchmarker,fzkn7.dbmsbenchmarker,w4cn7.dbmsbenchmarker
pod_count,1,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8
threads,64,8,8,8,8,8,8,8,8,64,8,8,8,8,8,8,8,8
target,32768,4096,4096,4096,4096,4096,4096,4096,4096,49152,6144,6144,6144,6144,6144,6144,6144,6144


### Add Metadata

In [24]:
df = collect.add_metadata(df_performance)
df.T

combine on columns phase


connection_pod,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2
code,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2
connection,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-8,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-4
configuration,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536
experiment_run,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
client,1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2
pod,g7bjq.dbmsbenchmarker,qf4pk.dbmsbenchmarker,khrjg.dbmsbenchmarker,tn66p.dbmsbenchmarker,rjcxw.dbmsbenchmarker,pr5fr.dbmsbenchmarker,97hvx.dbmsbenchmarker,944k6.dbmsbenchmarker,t9mf8.dbmsbenchmarker,jqtfk.dbmsbenchmarker,66kwz.dbmsbenchmarker,vmhc7.dbmsbenchmarker,8q8zp.dbmsbenchmarker,pctgl.dbmsbenchmarker,tp8ct.dbmsbenchmarker,kc9n2.dbmsbenchmarker,fzkn7.dbmsbenchmarker,w4cn7.dbmsbenchmarker
pod_count,1,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8
threads,64,8,8,8,8,8,8,8,8,64,8,8,8,8,8,8,8,8
target,32768,4096,4096,4096,4096,4096,4096,4096,4096,49152,6144,6144,6144,6144,6144,6144,6144,6144


# Performance Results per Total

In [25]:
df_performance = collect.get_performance_aggregated_per_phase()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)

In [26]:
df_performance.T

,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
configuration,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536
experiment_run,1,1,1,1
code,1776678767,1776678767,1776682364,1776682364
client,1,2,1,2
pod_count,1,8,1,8
threads,64,64,64,64
target,32768,32768,49152,49152
sf,3,3,3,3
workload,a,a,a,a


### With Metadata

In [27]:
df = collect.add_metadata(df_performance)
df.T

combine on index and column 'phase'


phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
configuration,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536
experiment_run,1,1,1,1
code,1776678767,1776678767,1776682364,1776682364
client,1,2,1,2
pod_count,1,8,1,8
threads,64,64,64,64
target,32768,32768,49152,49152
sf,3,3,3,3
workload,a,a,a,a
operations,3000000,3000000,3000000,3000000


In [28]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']]

,phase,code,configuration,experiment_run,client,type_tenants,num_tenants,vol_tenants
phase,,,,,,,,
1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-1,1776678767,1776678767-PostgreSQL-64-8-65536,1,1,None,0,False
1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767,1776678767-PostgreSQL-64-8-65536,1,2,None,0,False
1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-1,1776682364,1776682364-PostgreSQL-64-8-65536,1,1,None,0,False
1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364,1776682364-PostgreSQL-64-8-65536,1,2,None,0,False


In [29]:
collectors.get_non_constant(df).T

phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
configuration,1776678767-PostgreSQL-64-8-65536,1776678767-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536,1776682364-PostgreSQL-64-8-65536
code,1776678767,1776678767,1776682364,1776682364
client,1,2,1,2
pod_count,1,8,1,8
target,32768,32768,49152,49152
[OVERALL].RunTime(ms),1069952.00,1052642.00,1052107.00,1055137.00
[OVERALL].Throughput(ops/sec),2803.86,2867.72,2851.42,2871.07
[CLEANUP].AverageLatency(us),185.94,275.97,181.23,270.61
[CLEANUP].MinLatency(us),97.00,131.00,102.00,119.00
[CLEANUP].MaxLatency(us),737.00,1308.00,734.00,1317.00


# Performance at Ingestion

In [30]:
df_performance = collect.get_connections()
#df_performance = collect.get_loading_time_max_all()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-4,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-8
code,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776678767,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364,1776682364
connection,1776678767-PostgreSQL-64-8-65536-1-1,1776678767-PostgreSQL-64-8-65536-2-1,1776678767-PostgreSQL-64-8-65536-2-2,1776678767-PostgreSQL-64-8-65536-2-3,1776678767-PostgreSQL-64-8-65536-2-4,1776678767-PostgreSQL-64-8-65536-2-5,1776678767-PostgreSQL-64-8-65536-2-6,1776678767-PostgreSQL-64-8-65536-2-7,1776678767-PostgreSQL-64-8-65536-2-8,1776682364-PostgreSQL-64-8-65536-1-1,1776682364-PostgreSQL-64-8-65536-2-1,1776682364-PostgreSQL-64-8-65536-2-2,1776682364-PostgreSQL-64-8-65536-2-3,1776682364-PostgreSQL-64-8-65536-2-4,1776682364-PostgreSQL-64-8-65536-2-5,1776682364-PostgreSQL-64-8-65536-2-6,1776682364-PostgreSQL-64-8-65536-2-7,1776682364-PostgreSQL-64-8-65536-2-8
configuration,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536,PostgreSQL-64-8-65536
phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-2
experiment_run,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
client,1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2
dockerimage,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3
time_load,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1927.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00,1992.00
time_ingest,886.00,886.00,886.00,886.00,886.00,886.00,886.00,886.00,886.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00,905.00
time_check,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1040.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00,1086.00


# Hardware Monitoring for Loading Phase

In [31]:
df_performance = collect.get_monitoring_aggregated_per_phase("loading")
df_performance = collect.add_metadata(df_performance)
#df_performance_first = df_performance[df_performance['client']==1]
#df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

combine on index and column 'phase'


phase,1776678767-PostgreSQL-64-8-65536-1,1776678767-PostgreSQL-64-8-65536-2,1776682364-PostgreSQL-64-8-65536-1,1776682364-PostgreSQL-64-8-65536-2
Memory Usage [MiB],8293.92,8293.92,8319.30,8319.30
Memory Usage Cached [MiB],9549.27,9549.27,9581.76,9581.76
CPU Utilization,1.06,1.06,0.99,0.99
CPU Throttle,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0,0,0,0
CPU Utilization Time [s],1181.18,1181.18,1085.64,1085.64
CPU User Time [s],528.34,528.34,518.97,518.97
CPU System Time [s],652.84,652.84,566.66,566.66
Network Rx Total [MiB],3654.11,3654.11,3657.70,3657.70
Network Tx Total [MiB],3543.61,3543.61,3592.75,3592.75


In [32]:
#plot_bars(df_performance, y='CPU Utilization Time [s]', title='CPU [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [33]:
#plot_bars(df_performance, y='Memory Usage [MiB]', title='Memory Usage [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [34]:
#plot_bars(df_performance, y='Memory Usage Cached [MiB]', title='Memory Usage Cached [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Efficiency

In [35]:
client = 1

df_performance_monitoring = collect.get_monitoring_aggregated_per_phase(type="stream")
#df_performance_monitoring = df_performance_monitoring[df_performance_monitoring['client'] == client]
df_performance = collect.get_performance_aggregated_per_phase()
#df_performance = df_performance[df_performance['client'] == client]
merged_df = pd.merge(df_performance, df_performance_monitoring, left_index=True, right_index=True, how='left')
#merged_df = pd.merge(df_performance, df_performance_monitoring, on=['phase'], how='inner')
#merged_df['I_Lat'] = 1./merged_df['E_Lat']
merged_df = merged_df[merged_df['client'] == client]
merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']
merged_df.T

KeyError: 'Goodput (requests/second)'

In [ ]:
df = collect.add_metadata(merged_df)
df.T

In [ ]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']]